In [ ]:
import re
from collections import Counter
import torch
import torch.nn as nn

# ---------- Part A: Byte-Pair Encoding (BPE) ----------

def get_word_freqs(corpus):
    words = re.findall(r"\w+|[^\w\s]", corpus)
    return Counter(" ".join(list(w)) + " </w>" for w in words)

def get_pair_freqs(word_freqs):
    pairs = Counter()
    for word, freq in word_freqs.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_pair(pair, word_freqs):
    bigram = " ".join(pair)
    replacement = "".join(pair)
    new_word_freqs = {}
    for word, freq in word_freqs.items():
        new_word = word.replace(bigram, replacement)
        new_word_freqs[new_word] = freq
    return new_word_freqs

def train_bpe(corpus, num_merges=30):
    word_freqs = get_word_freqs(corpus)
    merges = []
    for _ in range(num_merges):
        pairs = get_pair_freqs(word_freqs)
        if not pairs:
            break
        best_pair = max(pairs, key=pairs.get)
        word_freqs = merge_pair(best_pair, word_freqs)
        merges.append(best_pair)
    vocab = sorted(set(sym for word in word_freqs for sym in word.split()))
    return merges, vocab

# ---------- Part B: Causal Language Model Training Loop ----------

class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size, d_model=32, n_heads=2, seq_len=16):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.ReLU(), nn.Linear(d_model * 2, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.token_emb(x) + self.pos_emb(pos)

        # custom causal mask -> prevents attending to future tokens
        causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()

        attn_out, _ = self.attn(h, h, h, attn_mask=causal_mask)
        h = self.ln1(h + attn_out)
        h = self.ln2(h + self.ff(h))
        return self.head(h)

def train_causal_lm(vocab_size, token_ids, epochs=5):
    model = TinyCausalLM(vocab_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    x = token_ids[:, :-1]
    y = token_ids[:, 1:]

    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        loss.backward()
        optimizer.step()
        print(f"epoch {epoch+1} loss {loss.item():.4f}")
    return model

if __name__ == "__main__":
    corpus = "the quick brown fox jumps over the lazy dog the dog barks"
    merges, vocab = train_bpe(corpus, num_merges=20)
    print("Learned", len(merges), "merges,", len(vocab), "vocab symbols")

    vocab_size = 50
    fake_ids = torch.randint(0, vocab_size, (4, 17))
    train_causal_lm(vocab_size, fake_ids, epochs=3)

---
## Task 2: Custom BPE Tokenizer & Autoregressive Causal LM